# Transfermarkt scraper — La Liga 2017–2025

1. **Rosters** — league page per season → team pages → one row per
   (season, club, player), with birth date and photo. `data/_rosters.parquet`.
2. **Profiles** — per row: nationality, height, foot, position.
   `data/_enrichment_checkpoint.csv`.
3. **Value histories** — per *player*, every market-value snapshot Transfermarkt
   has. `data/_value_histories.csv`.
4. **Season values** — each row's market value is the snapshot at the **end** of
   its season (June of year+1). Age is exact, from the birth date.
5. **Fill** — two-sided gaps in market value; ages without a birth date.

Why the end of the season: Understat stats cover the whole season (August to
May). A mid-season (December) value would be "explained" by goals scored after
it was set — the model would be learning from the future.

Every request goes through `fetch()` (retry, backoff, pacing), and stages 2 and
3 checkpoint to disk, so an interrupted run resumes instead of starting over.

In [1]:
import csv
import os
import random
import re
import time
from datetime import datetime
from pathlib import Path
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup


def find_project_root(marker: str = "CLAUDE.md") -> Path:
    """Walk up from the working directory until the repo root turns up."""
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / marker).exists():
            return candidate
    raise RuntimeError(f"could not find {marker} at or above {here}")


BASE_PATH = Path(os.environ.get("TRANSFER_EDGE_ROOT") or find_project_root())
DATA_DIR = BASE_PATH / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

ROSTER_PATH = DATA_DIR / "_rosters.parquet"              # stage 1 output
CHECKPOINT_PATH = DATA_DIR / "_enrichment_checkpoint.csv"  # stage 2 progress
PANEL_PATH = DATA_DIR / "players_panel.parquet"          # final output

YEARS = list(range(2017, 2026))   # saison_id 2017 == the 2017-18 season
TEAMS_PER_SEASON = 20             # La Liga; asserted, not assumed
REQUEST_DELAY = 0.2              # seconds between successful requests
MAX_RETRIES = 4

LEAGUE_URL = "https://www.transfermarkt.us/laliga/startseite/wettbewerb/ES1/plus/?saison_id={year}"
VALUE_API = "https://tmapi-alpha.transfermarkt.technology/player/{player_id}/market-value-history"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/142.0.0.0 Safari/537.36"
    )
}

SESSION = requests.Session()
SESSION.headers.update(HEADERS)

print(f"project root: {BASE_PATH}")

project root: /Users/luccagazotto/Documents/Personal/ML Projects/Soccer player/Soccer_player_market_value


In [2]:
class ScrapeError(RuntimeError):
    """A URL that still failed after every retry."""


def fetch(url: str, *, as_json: bool = False, timeout: int = 30):
    """GET with retry, exponential backoff and a pause between calls.

    Raises ScrapeError instead of returning None, so being rate-limited can
    never be mistaken for a player whose profile genuinely omits a field.
    """
    delay, last_error = 0.2, None
    for attempt in range(MAX_RETRIES):
        try:
            resp = SESSION.get(url, timeout=timeout)
            resp.raise_for_status()
            payload = resp.json() if as_json else BeautifulSoup(resp.content, "html.parser")
            time.sleep(REQUEST_DELAY)
            return payload
        except (requests.RequestException, ValueError) as exc:
            last_error = exc
            if attempt == MAX_RETRIES - 1:
                break
            time.sleep(delay + random.uniform(0, 0.5))  # jitter so retries desynchronise
            delay *= 2
    raise ScrapeError(f"{url} failed after {MAX_RETRIES} attempts: {last_error}")

## 1. Rosters

Club and player links are selected by their URL shape (`/startseite/verein/`,
`/profil/spieler/`) rather than by position in the table, so an extra promo row
on the page cannot shift the results the way a `[:-5][:20]` slice would.

In [3]:
def season_team_links(year: int) -> list[tuple[str, str]]:
    """Return [(club_name, club_url)] for one season, validated to 20 clubs."""
    url = LEAGUE_URL.format(year=year)
    soup = fetch(url)

    teams, seen = [], set()
    for a in soup.select("table.items td.hauptlink a[href*='/startseite/verein/']"):
        name = a.get_text(strip=True)
        if not name or name in seen:
            continue
        seen.add(name)
        teams.append((name, urljoin(url, a["href"])))

    if len(teams) != TEAMS_PER_SEASON:
        raise ScrapeError(
            f"{year}: expected {TEAMS_PER_SEASON} clubs, parsed {len(teams)} — "
            f"Transfermarkt's layout has probably changed. Got: {[t[0] for t in teams]}"
        )
    return teams


def parse_birth_date(text):
    """'Apr 30, 1992 (34)' -> '1992-04-30'. None if there is no date in it."""
    match = re.search(r"([A-Z][a-z]{2} \d{1,2}, \d{4})", text or "")
    if not match:
        return None
    try:
        return datetime.strptime(match.group(1), "%b %d, %Y").date().isoformat()
    except ValueError:
        return None


def portrait_url(img):
    """Full-size portrait URL from a squad-table <img>, or None.

    The table lazy-loads: the real URL sits in data-src and src is a 1x1 gif.
    Squad tables serve the 'medium' size; the profile header uses 'header',
    which is the same photo, larger. Transfermarkt's grey silhouette for
    players without a photo is dropped rather than stored.
    """
    if img is None:
        return None
    url = img.get("data-src") or img.get("src") or ""
    if not url.startswith("http") or "/portrait/" not in url or "default" in url:
        return None
    return url.replace("/portrait/medium/", "/portrait/header/")


def parse_team_page(soup, team_url: str) -> list[dict]:
    """One dict per player row in a squad table. Deduplicated by player_id."""
    players, seen = [], set()
    for tr in soup.select("table.items > tbody > tr"):
        a = tr.select_one("td.hauptlink a[href*='/profil/spieler/']")
        if a is None:
            continue
        href = urljoin(team_url, a["href"])
        match = re.search(r"/profil/spieler/(\d+)", href)
        if not match or match.group(1) in seen:
            continue
        seen.add(match.group(1))
        birth = next((parse_birth_date(td.get_text(" ", strip=True))
                      for td in tr.find_all("td", recursive=False)
                      if parse_birth_date(td.get_text(" ", strip=True))), None)
        players.append({
            "player_id": match.group(1),
            "player_name": a.get_text(strip=True),
            "link_html": href,
            "birth_date": birth,
            "image_url": portrait_url(tr.select_one("img.bilderrahmen-fixed")),
        })
    return players


def scrape_rosters(years=YEARS) -> pd.DataFrame:
    """One row per (season, club, player), with birth date and photo URL."""
    rows = []
    for year in years:
        teams = season_team_links(year)
        before = len(rows)
        for club, team_url in teams:
            for p in parse_team_page(fetch(team_url), team_url):
                rows.append({"year": year, "club": club, **p})
        print(f"{year}: {len(teams)} clubs, {len(rows) - before} players")
    return pd.DataFrame(rows)


def add_roster_details(roster: pd.DataFrame) -> pd.DataFrame:
    """Add birth_date / image_url to a roster cached before they were scraped.

    Re-reads only the team pages (~190 requests), never the player profiles,
    and leaves every existing roster row as it was.
    """
    found = []
    for year in sorted(roster["year"].astype(int).unique()):
        for club, team_url in season_team_links(int(year)):
            for p in parse_team_page(fetch(team_url), team_url):
                found.append({"year": int(year), "player_id": p["player_id"],
                              "birth_date": p["birth_date"], "image_url": p["image_url"]})
        print(f"{year}: details read")
    details = pd.DataFrame(found).drop_duplicates(subset=["year", "player_id"])
    return (roster.drop(columns=["birth_date", "image_url"], errors="ignore")
            .astype({"year": int, "player_id": str})
            .merge(details, on=["year", "player_id"], how="left"))

In [4]:
if ROSTER_PATH.exists():
    roster = pd.read_parquet(ROSTER_PATH)
    print(f"reusing {ROSTER_PATH.name}: {roster.shape}")
else:
    roster = scrape_rosters()
    roster.to_parquet(ROSTER_PATH, index=False)
    print(f"saved {ROSTER_PATH.name}: {roster.shape}")

# Rosters cached before photos / birth dates were scraped get them added here —
# team pages only, so the (expensive) enrichment checkpoint stays valid.
if not {"birth_date", "image_url"} <= set(roster.columns):
    roster = add_roster_details(roster)
    roster.to_parquet(ROSTER_PATH, index=False)
    print(f"added birth_date / image_url -> {ROSTER_PATH.name}: {roster.shape}")

print(f"with birth date: {roster.birth_date.notna().mean():.0%}   "
      f"with photo: {roster.image_url.notna().mean():.0%}")
roster.head()

reusing _rosters.parquet: (6677, 7)
with birth date: 100%   with photo: 94%


,year,club,player_id,player_name,link_html,birth_date,image_url
0,2017,FC Barcelona,74857,Marc ter Stegen,https://www.transfermarkt.us/marc-ter-stegen/p...,1992-04-30,https://img.a.transfermarkt.technology/portrai...
1,2017,FC Barcelona,146227,Jasper Cillessen,https://www.transfermarkt.us/jasper-cillessen/...,1989-04-22,https://img.a.transfermarkt.technology/portrai...
2,2017,FC Barcelona,142033,Adrián Ortolá,https://www.transfermarkt.us/adrian-ortola/pro...,1993-08-20,https://img.a.transfermarkt.technology/portrai...
3,2017,FC Barcelona,126540,Samuel Umtiti,https://www.transfermarkt.us/samuel-umtiti/pro...,1993-11-14,https://img.a.transfermarkt.technology/portrai...
4,2017,FC Barcelona,18944,Gerard Piqué,https://www.transfermarkt.us/gerard-pique/prof...,1987-02-02,https://img.a.transfermarkt.technology/portrai...


## 2. Profiles

One request per roster row: the profile page for nationality, height, foot and
position. Each completed row is appended to `_enrichment_checkpoint.csv` and
flushed, so a crash or a ban costs only the row in flight; rows that fail every
retry are not checkpointed, so re-running the cell retries exactly those.

Market value and age are no longer taken here — see sections 3 and 4.

In [5]:
def info_table_value(soup, label: str):
    """Read one value out of Transfermarkt's info table by its label."""
    for span in soup.select("span.info-table__content--regular"):
        if span.get_text(strip=True) == label:
            sibling = span.find_next_sibling("span", class_="info-table__content--bold")
            if sibling:
                return sibling.get_text(strip=True)
    return None


def parse_height_m(text: str):
    """Parse a height string to metres.

    transfermarkt.us renders height as "6 ft 2 in" (imperial), not the "1,87 m"
    (metric, comma decimal) the .com/.de domains use. Handle both, since which
    one you get can depend on the domain and may change again.
    """
    if not text:
        return None
    text = text.strip()

    imperial = re.match(r"(\d+)\s*ft\s*(\d+)\s*in", text)
    if imperial:
        feet, inches = int(imperial.group(1)), int(imperial.group(2))
        return round((feet * 12 + inches) * 0.0254, 2)

    metric = re.match(r"([\d,.]+)\s*m", text)
    if metric:
        try:
            return float(metric.group(1).replace(",", "."))
        except ValueError:
            return None

    return None


def parse_profile(soup) -> dict:
    """Static attributes from a profile page; None where Transfermarkt omits one.

    Every field here is genuinely optional. Transport failures raise out of
    fetch() instead, which is what keeps "not listed" and "we got blocked"
    distinguishable.
    """
    nationality = soup.select_one('span[itemprop="nationality"]')

    height = None
    height_tag = soup.select_one('span[itemprop="height"]')
    if height_tag:
        height = parse_height_m(height_tag.get_text(strip=True))

    return {
        "nationality": nationality.get_text(strip=True) if nationality else None,
        "height": height,
        "foot": info_table_value(soup, "Foot:"),
        "position": info_table_value(soup, "Position:"),
    }

In [6]:
HISTORY_PATH = DATA_DIR / "_value_histories.csv"   # stage 3 progress
HISTORY_COLS = ["player_id", "date", "market_value_eur", "club_id"]


def fetch_value_histories(player_ids) -> pd.DataFrame:
    """Every market-value snapshot for every player — one request per player.

    Checkpointed like the profiles: a player whose history is empty still gets
    a marker row (no date), so a re-run doesn't ask for him again.
    """
    done = set()
    if HISTORY_PATH.exists():
        done = set(pd.read_csv(HISTORY_PATH, dtype={"player_id": str}, usecols=["player_id"]).player_id)
    todo = [p for p in dict.fromkeys(map(str, player_ids)) if p not in done]
    print(f"{len(done)} players cached, {len(todo)} to fetch (~{len(todo) * REQUEST_DELAY / 60:.0f} min)")

    failures = []
    write_header = not HISTORY_PATH.exists()
    with HISTORY_PATH.open("a", newline="") as fh:
        writer = csv.DictWriter(fh, fieldnames=HISTORY_COLS)
        if write_header:
            writer.writeheader()
        for n, pid in enumerate(todo, 1):
            try:
                payload = fetch(VALUE_API.format(player_id=pid), as_json=True) or {}
            except ScrapeError as exc:
                failures.append(pid)
                print(f"  ! {pid}: {exc}")
                continue                      # not checkpointed, so a re-run retries it
            snapshots = [
                {"player_id": pid,
                 "date": h["marketValue"]["determined"],
                 "market_value_eur": h["marketValue"]["value"],
                 "club_id": h.get("clubId")}
                for h in (payload.get("data", {}).get("history") or [])
                if h.get("marketValue", {}).get("determined")
            ]
            writer.writerows(snapshots or [{"player_id": pid}])
            fh.flush()
            if n % 200 == 0:
                print(f"  {n}/{len(todo)}")

    if failures:
        print(f"\n{len(failures)} players failed — re-run this cell to retry them")
    hist = pd.read_csv(HISTORY_PATH, dtype={"player_id": str, "club_id": str})
    hist["date"] = pd.to_datetime(hist["date"], errors="coerce")
    return hist.dropna(subset=["date"])


def value_at_season_end(hist: pd.DataFrame, year: int):
    """The snapshot closest to the end of season `year` (which ends in year+1).

    June of year+1 first — Transfermarkt's big post-season update — then the
    last one in May, then the first one in July. Returns (value, date), or
    (None, None) when none of those months has a snapshot.
    """
    if hist is None or hist.empty:
        return None, None
    end = year + 1
    for month, which in ((6, "last"), (5, "last"), (7, "first")):
        hits = hist[(hist["date"].dt.year == end) & (hist["date"].dt.month == month)]
        if hits.empty:
            continue
        row = hits.sort_values("date").iloc[-1 if which == "last" else 0]
        return float(row["market_value_eur"]), row["date"]
    return None, None


def age_at_season_end(birth_date: pd.Series, year: pd.Series) -> pd.Series:
    """Exact age on 30 June of year+1 — the date the season's value is taken."""
    born = pd.to_datetime(birth_date, errors="coerce")
    not_yet = (born.dt.month > 6).astype(float)   # birthday after 30 June: not had it yet
    return (year.astype(int) + 1) - born.dt.year - not_yet

In [7]:
FIELDS = ["year", "player_id", "nationality", "height", "foot", "position"]


def enrich(roster: pd.DataFrame) -> pd.DataFrame:
    """Fetch the profile for every roster row, checkpointing as it goes."""
    done, header = set(), FIELDS
    if CHECKPOINT_PATH.exists():
        prior = pd.read_csv(CHECKPOINT_PATH, dtype={"player_id": str})
        done = set(zip(prior.year.astype(int), prior.player_id))
        header = list(prior.columns)   # older checkpoints carry extra columns; keep appends aligned
        print(f"resuming — {len(done)} rows already fetched")

    todo = [r for r in roster.itertuples(index=False)
            if (int(r.year), str(r.player_id)) not in done]
    print(f"{len(todo)} rows to fetch (~{len(todo) * REQUEST_DELAY / 60:.0f} min)")

    failures = []
    with CHECKPOINT_PATH.open("a", newline="") as fh:
        writer = csv.DictWriter(fh, fieldnames=header, extrasaction="ignore")
        if not done:
            writer.writeheader()

        for n, r in enumerate(todo, 1):
            record = {"year": int(r.year), "player_id": str(r.player_id)}
            try:
                record.update(parse_profile(fetch(r.link_html)))
            except ScrapeError as exc:
                failures.append((r.year, r.player_id, r.player_name))
                print(f"  ! {r.player_name} ({r.year}): {exc}")
                continue                      # not checkpointed, so a re-run retries it
            writer.writerow(record)
            fh.flush()
            if n % 100 == 0:
                print(f"  {n}/{len(todo)}")

    if failures:
        print(f"\n{len(failures)} rows failed — re-run this cell to retry them")

    fetched = (
        pd.read_csv(CHECKPOINT_PATH, dtype={"player_id": str})
        .drop_duplicates(subset=["year", "player_id"], keep="last")
        [FIELDS]    # a December market_value / age from older runs is ignored on purpose
    )
    return (
        roster.astype({"year": int, "player_id": str})
        .merge(fetched, on=["year", "player_id"], how="left")
    )


panel = enrich(roster)
print(f"\nenriched panel: {panel.shape}")
panel.head()

resuming — 6513 rows already fetched
0 rows to fetch (~0 min)

enriched panel: (6677, 11)


,year,club,player_id,player_name,link_html,birth_date,image_url,nationality,height,foot,position
0,2017,FC Barcelona,74857,Marc ter Stegen,https://www.transfermarkt.us/marc-ter-stegen/p...,1992-04-30,https://img.a.transfermarkt.technology/portrai...,Germany,1.88,right,Goalkeeper
1,2017,FC Barcelona,146227,Jasper Cillessen,https://www.transfermarkt.us/jasper-cillessen/...,1989-04-22,https://img.a.transfermarkt.technology/portrai...,Netherlands,1.88,right,Goalkeeper
2,2017,FC Barcelona,142033,Adrián Ortolá,https://www.transfermarkt.us/adrian-ortola/pro...,1993-08-20,https://img.a.transfermarkt.technology/portrai...,Spain,1.88,left,Goalkeeper
3,2017,FC Barcelona,126540,Samuel Umtiti,https://www.transfermarkt.us/samuel-umtiti/pro...,1993-11-14,https://img.a.transfermarkt.technology/portrai...,France,1.83,left,Defender - Centre-Back
4,2017,FC Barcelona,18944,Gerard Piqué,https://www.transfermarkt.us/gerard-pique/prof...,1987-02-02,https://img.a.transfermarkt.technology/portrai...,Spain,1.93,right,Defender - Centre-Back


## 3. Value histories

One request per player (not per season): the API returns the player's whole
history in one go, so asking again for every season he played was wasted work.

In [8]:
histories = fetch_value_histories(roster["player_id"].unique())
print(f"\n{histories.player_id.nunique()} players with a history, {len(histories):,} snapshots")
histories.head()

0 players cached, 2586 to fetch (~9 min)
  200/2586
  400/2586
  600/2586
  800/2586
  1000/2586
  1200/2586
  1400/2586
  1600/2586
  1800/2586
  2000/2586
  2200/2586
  2400/2586

2559 players with a history, 57,785 snapshots


,player_id,date,market_value_eur,club_id
0,74857,2009-08-13,100000.0,4121
1,74857,2010-08-27,200000.0,1006
2,74857,2010-10-18,350000.0,18
3,74857,2011-01-12,400000.0,18
4,74857,2011-06-29,2500000.0,18


## 4. Season values and ages

Each row takes the snapshot at the end of its season (`value_date` records
which one). Age is computed from the birth date on the same date, so it is exact
and consistent with the value.

In [9]:
by_player = {pid: g for pid, g in histories.groupby("player_id")}
season_values = [value_at_season_end(by_player.get(str(r.player_id)), int(r.year))
                 for r in panel.itertuples(index=False)]
panel["market_value"] = [v for v, _ in season_values]
panel["value_date"] = [dt for _, dt in season_values]
panel["age"] = age_at_season_end(panel["birth_date"], panel["year"])

print(f"rows with an end-of-season value: {panel.market_value.notna().mean():.0%}")
print("month the value was taken from:")
print(pd.to_datetime(panel.value_date).dt.month.value_counts().rename({5: "May", 6: "June", 7: "July"}).to_string())
print(f"rows with an age: {panel.age.notna().mean():.0%}")

rows with an end-of-season value: 86%
month the value was taken from:
value_date
June    4930
May      680
July     100
rows with an age: 100%


## 5. Fill missing market values and ages

**Market value.** A gap is filled with the **mean of the nearest season before
and the nearest season after it in which he has a listed value** — and only
when both sides exist. A gap at the start or end of his record stays empty (and
the row drops out at `build_features`): copying a neighbour's value there would
put a later valuation into an earlier season.

Filled rows are flagged in `market_value_imputed` and are **never used by the
model** — not as training targets, not as test rows, not as lags. The midpoint
is built from the *next* season's value, so using it anywhere in the model would
leak the future. They exist only so the app's value history has no holes.

**Age.** Already exact from the birth date (section 4). The few rows without a
birth date take it from a neighbouring season plus the year difference.

In [10]:
def fill_missing_values(df: pd.DataFrame) -> pd.DataFrame:
    """Fill market_value (two-sided midpoint only) and ages with no birth date."""
    df = df.copy()
    df["year"] = df["year"].astype(int)
    for col in ("age", "market_value"):
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df = df.sort_values(["player_id", "year"]).reset_index(drop=True)

    # Market value: midpoint of the nearest listed season on each side. NaN
    # unless both exist — deliberately no one-sided fallback (see markdown).
    value_missing = df["market_value"].isna()
    by_player = df.groupby("player_id")
    midpoint = (by_player["market_value"].ffill() + by_player["market_value"].bfill()) / 2
    df["market_value"] = df["market_value"].fillna(midpoint)
    df["market_value_imputed"] = value_missing & df["market_value"].notna()

    # Age: a neighbouring season's age plus the year difference (exact too).
    age_missing = df["age"].isna()
    df["_known_age"] = df["age"]
    df["_known_year"] = df["year"].where(df["age"].notna())
    by_player = df.groupby("player_id")
    from_prev = by_player["_known_age"].ffill() + (df["year"] - by_player["_known_year"].ffill())
    from_next = by_player["_known_age"].bfill() - (by_player["_known_year"].bfill() - df["year"])
    df["age"] = df["age"].fillna(from_prev).fillna(from_next)
    df["age_imputed"] = age_missing & df["age"].notna()

    return df.drop(columns=["_known_age", "_known_year"])


n_missing_value = int(panel["market_value"].isna().sum())
n_missing_age = int(panel["age"].isna().sum())
panel = fill_missing_values(panel)

print(f"market values missing : {n_missing_value}")
print(f"  filled (two-sided)  : {int(panel.market_value_imputed.sum())}  (display only, never modelled)")
print(f"  left empty          : {int(panel.market_value.isna().sum())}")
print(f"ages missing          : {n_missing_age}")
print(f"  still missing       : {int(panel.age.isna().sum())}")

market values missing : 967
  filled (two-sided)  : 320  (display only, never modelled)
  left empty          : 647
ages missing          : 0
  still missing       : 0


In [11]:
panel.to_parquet(PANEL_PATH, index=False)
print(f"saved {PANEL_PATH.name}: {panel.shape}")
print(f"columns: {list(panel.columns)}")
panel.head()

saved players_panel.parquet: (6677, 16)
columns: ['year', 'club', 'player_id', 'player_name', 'link_html', 'birth_date', 'image_url', 'nationality', 'height', 'foot', 'position', 'market_value', 'value_date', 'age', 'market_value_imputed', 'age_imputed']


,year,club,player_id,player_name,link_html,birth_date,image_url,nationality,height,foot,position,market_value,value_date,age,market_value_imputed,age_imputed
0,2023,Valencia CF,1000135,Joselu Pérez,https://www.transfermarkt.us/joselu-perez/prof...,2004-03-12,None,Spain,1.83,right,Attack - Centre-Forward,100000.0,2024-06-24,20.0,False,False
1,2025,Getafe CF,1000135,Joselu Pérez,https://www.transfermarkt.us/joselu-perez/prof...,2004-03-12,None,Spain,1.83,right,Attack - Centre-Forward,200000.0,2026-06-08,22.0,False,False
2,2022,Getafe CF,1000136,Gorka Rivera,https://www.transfermarkt.us/gorka-rivera/prof...,2004-08-01,https://img.a.transfermarkt.technology/portrai...,Spain,NaN,left,Defender - Left-Back,NaN,NaT,18.0,False,False
3,2023,Getafe CF,1000136,Gorka Rivera,https://www.transfermarkt.us/gorka-rivera/prof...,2004-08-01,https://img.a.transfermarkt.technology/portrai...,Spain,NaN,left,Defender - Left-Back,100000.0,2024-06-24,19.0,False,False
4,2024,Getafe CF,1000136,Gorka Rivera,https://www.transfermarkt.us/gorka-rivera/prof...,2004-08-01,https://img.a.transfermarkt.technology/portrai...,Spain,NaN,left,Defender - Left-Back,100000.0,2025-06-25,20.0,False,False
